<div dir="rtl" style="
    text-align: center;
    background-color: #2E7D32;
    color: #FF0000;
    padding: 15px;
    border-radius: 10px;
">
    پاسخنامه کوییز یادگیری ماشین
    <br>
    نویسنده : احسان سویزی
    <br>
</div>

In [22]:
# import requiered libraries
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
pd.set_option("display.max_columns", None)

In [23]:
import json
from pathlib import Path

from data_factory.config import load_config
from data_factory.ingestion import (
    build_scope_filter,
    count_records,
    download_batches,
    prepare_checkpoint,
)

# Support opening the notebook from the project or its parent directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config/setting.json").is_file():
    PROJECT_ROOT = PROJECT_ROOT / "machine_learning_challenge"
if not (PROJECT_ROOT / "config/setting.json").is_file():
    raise FileNotFoundError("Open this notebook from the project directory.")

config = load_config(PROJECT_ROOT / "config/setting.json")
config["paths"]["raw_dir"] = str(PROJECT_ROOT / config["paths"]["raw_dir"])
source = config["source"]
http = config["http"]

print(source)


{'endpoint': 'https://data.cityofnewyork.us/resource/erm2-nwe9.json', 'start_date': '2024-01-01T00:00:00', 'end_date_exclusive': '2025-07-01T00:00:00', 'boroughs': [], 'complaint_types': ['Noise - Residential', 'Noise - Vehicle', 'Noise - Street/Sidewalk', 'Noise - Park', 'Noise - Commercial'], 'minimum_records': 300000, 'seed': 42}


In [26]:
scope_filter = build_scope_filter(config)
total_records = count_records(config)
minimum_records = source["minimum_records"]

print("Filter:", scope_filter)
print("Matching records:", f"{total_records:,}")
print("Minimum requirement met:", total_records >= minimum_records)


Filter: created_date >= '2024-01-01T00:00:00.000000' AND created_date < '2025-07-01T00:00:00.000000' AND complaint_type IN ('Noise - Commercial', 'Noise - Park', 'Noise - Residential', 'Noise - Street/Sidewalk', 'Noise - Vehicle')
Matching records: 1,017,635
Minimum requirement met: True


<div dir="rtl" style="text-align: right; line-height: 2;">

تنظیمات منبع در فایل `config/setting.json` نگهداری می‌شوند. بازهٔ انتخاب‌شده از ابتدای ژانویهٔ ۲۰۲۴ تا ابتدای ژوئیهٔ ۲۰۲۵ است؛ تاریخ پایان شامل نمی‌شود و بازه دقیقاً ۱۸ ماه پیوسته است.

پنج نوع شکایت `Noise - Residential`، `Noise - Vehicle`، `Noise - Street/Sidewalk`، `Noise - Park` و `Noise - Commercial` از همهٔ مناطق شهر انتخاب شده‌اند. تعداد زنده در خروجی سلول قبل نمایش داده می‌شود؛ شمارش اولیه برابر با ۱٬۰۱۷٬۶۳۵ رکورد بود.

عدد ۳۰۰٬۰۰۰ حداقل موردنیاز است و سقف دانلود نیست. نمونه‌گیری انجام نمی‌دهیم و تمام رکوردهای مطابق این فیلتر را دریافت می‌کنیم. مقدار `seed` در دانلود فعلی کاربردی ندارد.

</div>


<div dir="rtl" style="text-align: right; line-height: 2;">

**آماده‌سازی checkpoint**

سلول بعد، بدون درخواست به API، وضعیت موجود را بررسی می‌کند و در صورت نبودن آن، فایل `data/raw_data/checkpoint.json` را می‌سازد. این فایل شمارهٔ آخرین batch، تعداد رکوردهای ذخیره‌شده، آخرین `created_date` و `unique_key`، مشخصات فیلتر و هش فایل‌ها را نگه می‌دارد. آن را دستی ویرایش نکنید.

رکوردها بر اساس `created_date ASC, unique_key ASC` مرتب می‌شوند. شرط درخواست بعدی از هر دو مقدار استفاده می‌کند تا رکوردهای هم‌زمان جا نیفتند؛ `unique_key` رشته باقی می‌ماند.

</div>


In [28]:
checkpoint = prepare_checkpoint(config)
checkpoint_path = Path(config["paths"]["raw_dir"]) / config["paths"]["checkpoint_file"]

print("Checkpoint:", checkpoint_path.relative_to(PROJECT_ROOT))
print(json.dumps(
    {key: checkpoint[key] for key in (
        "last_batch", "downloaded_records", "last_created_date",
        "last_unique_key", "completed",
    )},
    indent=2,
))


Checkpoint: data\raw_data\checkpoint.json
{
  "last_batch": 0,
  "downloaded_records": 0,
  "last_created_date": null,
  "last_unique_key": null,
  "completed": false
}


<div dir="rtl" style="text-align: right; line-height: 2;">

**دانلود خودکار و ادامه پس از توقف**

سلول بعد را یک بار اجرا کنید؛ حلقه، batchهای ۵۰۰۰تایی را پشت سر هم دریافت می‌کند و فایل‌های `batch_000001.json`، `batch_000002.json` و … را خودکار می‌سازد. مقدار batch از تنظیمات خوانده می‌شود. برای هر batch نیازی به تغییر کد نیست.

ابتدا فایل batch به‌صورت اتمی ذخیره می‌شود و سپس checkpoint ثبت می‌شود. اگر اجرا بین این دو مرحله قطع شود، اجرای بعدی batch ذخیره‌شده را بازیابی می‌کند. خطاهای موقت شبکه با تعداد تلاش تنظیم‌شده تکرار می‌شوند. در صورت خطای نهایی یا توقف با Interrupt، همان سلول را دوباره اجرا کنید؛ از آخرین پیشرفت ذخیره‌شده ادامه می‌دهد.

حلقه تا دریافت پاسخ خالی ادامه دارد؛ کوچک بودن یک batch به‌تنهایی نشانهٔ پایان نیست. وقتی `completed` برابر `true` باشد، اجرای مجدد همین snapshot درخواست شبکه‌ای نمی‌فرستد. این وضعیت به معنی دریافت تغییرات بعدی منبع زنده نیست. اگر فیلتر را تغییر دادید، در تنظیمات یک مسیر `raw_dir` جدا انتخاب کنید؛ داده‌های قدیمی را با دامنهٔ جدید مخلوط نمی‌کنیم.

در هر لحظه فقط یک batch در حافظه است. فایل‌های خام و checkpoint در Git نادیده گرفته می‌شوند توسط فایل .gitignore. خروجی دانلود کامل فقط بعد از اجرای واقعی سلول زیر ایجاد می‌شود.

</div>


In [30]:
# One invocation downloads all remaining batches automatically.
# Interrupt safely, then rerun this same cell to resume.
download_summary = download_batches(config)

print("Downloaded records:", f"{download_summary['downloaded_records']:,}")
print("Saved batches:", download_summary["last_batch"])
print("Completed:", download_summary["completed"])


Saved batch_000044.json: 5,000 records | total 220,000 | checkpoint updated
Saved batch_000045.json: 5,000 records | total 225,000 | checkpoint updated
Saved batch_000046.json: 5,000 records | total 230,000 | checkpoint updated
Saved batch_000047.json: 5,000 records | total 235,000 | checkpoint updated
Saved batch_000048.json: 5,000 records | total 240,000 | checkpoint updated
Saved batch_000049.json: 5,000 records | total 245,000 | checkpoint updated
Saved batch_000050.json: 5,000 records | total 250,000 | checkpoint updated
Saved batch_000051.json: 5,000 records | total 255,000 | checkpoint updated
Saved batch_000052.json: 5,000 records | total 260,000 | checkpoint updated
Saved batch_000053.json: 5,000 records | total 265,000 | checkpoint updated
Saved batch_000054.json: 5,000 records | total 270,000 | checkpoint updated
Saved batch_000055.json: 5,000 records | total 275,000 | checkpoint updated
Saved batch_000056.json: 5,000 records | total 280,000 | checkpoint updated
Saved batch_

KeyboardInterrupt: 

In [ ]:
# Read the persisted state, including after an interrupted download.
checkpoint = prepare_checkpoint(config)
print("Saved records:", f"{checkpoint['downloaded_records']:,}")
print("Saved batches:", checkpoint["last_batch"])
print("Completed:", checkpoint["completed"])

if checkpoint["last_batch"]:
    # Preview one saved batch, never concatenate the entire raw dataset.
    first_batch_path = Path(config["paths"]["raw_dir"]) / "batch_000001.json"
    with first_batch_path.open(encoding="utf-8") as file:
        first_batch = json.load(file)
    display(pd.DataFrame(first_batch[:5]))
    del first_batch


Saved records: 310,000
Saved batches: 62
Completed: False


,address_type,agency,agency_name,bbl,borough,city,closed_date,community_board,complaint_type,council_district,created_date,cross_street_1,cross_street_2,descriptor,incident_address,incident_zip,intersection_street_1,intersection_street_2,landmark,latitude,location,location_type,longitude,open_data_channel_type,park_borough,park_facility_name,police_precinct,resolution_action_updated_date,resolution_description,status,street_name,unique_key,x_coordinate_state_plane,y_coordinate_state_plane
0,ADDRESS,NYPD,New York City Police Department,2032367501,BRONX,BRONX,2024-01-01T00:59:31.000,07 BRONX,Noise - Residential,14,2024-01-01T00:00:20.000,CEDAR AVENUE,MAJOR DEEGAN EXPRESSWAY,Loud Music/Party,233 LANDING ROAD,10468,CEDAR AVENUE,MAJOR DEEGAN EXPRESSWAY,LANDING ROAD,40.86240645467484,"{'coordinates': [-73.911097148279, 40.86240645...",Residential Building/House,-73.91109714827905,PHONE,BRONX,Unspecified,Precinct 52,2024-01-01T00:59:34.000,The Police Department responded to the complai...,Closed,LANDING ROAD,59889016,1008841,253488
1,ADDRESS,NYPD,New York City Police Department,3043710001,BROOKLYN,BROOKLYN,2024-01-01T00:56:45.000,05 BROOKLYN,Noise - Residential,42,2024-01-01T00:00:43.000,VERMONT STREET,VAN SICLEN AVENUE,Banging/Pounding,640 STANLEY AVENUE,11207,VERMONT STREET,VAN SICLEN AVENUE,STANLEY AVENUE,40.65893182677183,"{'coordinates': [-73.884862783149, 40.65893182...",Residential Building/House,-73.88486278314905,MOBILE,BROOKLYN,Unspecified,Precinct 75,2024-01-01T00:56:50.000,The Police Department responded to the complai...,Closed,STANLEY AVENUE,59888940,1016195,179364
2,ADDRESS,NYPD,New York City Police Department,4013110066,QUEENS,WOODSIDE,2024-01-01T00:48:34.000,02 QUEENS,Noise - Street/Sidewalk,26,2024-01-01T00:00:48.000,41 AVENUE,WOODSIDE AVENUE,Loud Music/Party,41-52 72 STREET,11377,41 AVENUE,WOODSIDE AVENUE,72 STREET,40.74376141146886,"{'coordinates': [-73.893186092905, 40.74376141...",Street/Sidewalk,-73.8931860929054,ONLINE,QUEENS,Unspecified,Precinct 108,2024-01-01T00:48:39.000,The Police Department responded to the complai...,Closed,72 STREET,59892651,1013848,210267
3,ADDRESS,NYPD,New York City Police Department,4090210002,QUEENS,OZONE PARK,2024-01-01T00:29:32.000,09 QUEENS,Noise - Street/Sidewalk,32,2024-01-01T00:00:51.000,85 STREET,86 STREET,Loud Music/Party,85-04 95 AVENUE,11416,85 STREET,86 STREET,95 AVENUE,40.68473836098068,"{'coordinates': [-73.855753236702, 40.68473836...",Street/Sidewalk,-73.85575323670186,MOBILE,QUEENS,Unspecified,Precinct 102,2024-01-01T00:29:34.000,The Police Department responded to the complai...,Closed,95 AVENUE,59889054,1024256,188778
4,ADDRESS,NYPD,New York City Police Department,4150010087,QUEENS,JAMAICA,2024-01-01T01:32:22.000,13 QUEENS,Noise - Street/Sidewalk,31,2024-01-01T00:01:01.000,SOUTH CONDUIT AVENUE,146 AVENUE,Loud Music/Party,144-25 153 LANE,11434,SOUTH CONDUIT AVENUE,146 AVENUE,153 LANE,40.66610329232783,"{'coordinates': [-73.78495540739, 40.666103292...",Street/Sidewalk,-73.78495540739017,MOBILE,QUEENS,Unspecified,Precinct 105,2024-01-01T01:32:26.000,The Police Department responded to the complai...,Closed,153 LANE,59892073,1043908,182029
